# Tarea 3. Exploración y análisis de datos
## Pregunta 3

En esta etapa se realiza la exploración y análisis de la base analítica construida durante la etapa de alistamiento de datos de la Pregunta 3.

El objetivo es identificar patrones, concentraciones y relaciones relevantes entre las características contractuales, financieras, temporales e institucionales de los contratos, con el propósito de establecer hallazgos que posteriormente orienten el diseño del tablero.

## 1. Carga de la base analítica

Se utiliza como insumo la base analítica generada en la Tarea 2, Pregunta 3. Esta base contiene los registros seleccionados y los indicadores construidos durante las etapas de limpieza y alistamiento.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../Tarea 2/Pregunta 3/df_analitico_p3.csv")

print("Base analítica cargada correctamente.")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Base analítica cargada correctamente.
Filas: 20718
Columnas: 31


## 2. Caracterización general de los contratos

Como primera aproximación exploratoria se analiza la distribución de los contratos según su estado, tipo de contrato y modalidad de contratación.

Esta caracterización permite establecer el contexto general de la base e identificar las categorías con mayor presencia, que posteriormente podrán utilizarse para profundizar en el comportamiento financiero y contractual.

In [2]:
# Distribución de contratos por estado

contratos_estado = (
    df["estado_contrato"]
    .value_counts()
    .rename_axis("estado_contrato")
    .reset_index(name="contratos")
)

contratos_estado

,estado_contrato,contratos
0,Cerrado,7622
1,Modificado,4132
2,terminado,2946
3,Borrador,2131
4,En ejecución,1720
5,Aprobado,773
6,Cancelado,578
7,En aprobación,283
8,enviado Proveedor,250
9,Suspendido,222


In [3]:
# Distribución de contratos por tipo

contratos_tipo = (
    df["tipo_de_contrato"]
    .value_counts()
    .rename_axis("tipo_de_contrato")
    .reset_index(name="contratos")
)

contratos_tipo

,tipo_de_contrato,contratos
0,Prestación de servicios,12058
1,Obra,3409
2,Otro,2326
3,Interventoría,1530
4,Suministros,593
5,Consultoría,573
6,Compraventa,81
7,Comodato,74
8,Seguros,32
9,Arrendamiento de inmuebles,31


In [4]:
# Distribución de contratos por modalidad de contratación

contratos_modalidad = (
    df["modalidad_de_contratacion"]
    .value_counts()
    .rename_axis("modalidad_de_contratacion")
    .reset_index(name="contratos")
)

contratos_modalidad

,modalidad_de_contratacion,contratos
0,Contratación directa,14377
1,Mínima cuantía,2909
2,Concurso de méritos abierto,1297
3,Licitación pública Obra Publica,787
4,Selección Abreviada de Menor Cuantía,724
5,Licitación pública,429
6,Selección abreviada subasta inversa,101
7,Seleccion Abreviada Menor Cuantia Sin Manifest...,52
8,Contratación Directa (con ofertas),33
9,Contratación régimen especial,8


## 3. Distribución del valor contractual

El valor de los contratos presenta una alta dispersión y la presencia de valores extremos identificados durante la etapa de limpieza.

Por esta razón, se utilizan medidas robustas como la mediana y los percentiles para caracterizar la distribución, evitando que los valores extremos dominen la interpretación del comportamiento general.

In [5]:
# Estadísticos principales del valor contractual

valor_contrato = df["valor_del_contrato"]

print(f"Valor total contractual: ${valor_contrato.sum():,.0f}")
print(f"Valor promedio: ${valor_contrato.mean():,.0f}")
print(f"Valor mediano: ${valor_contrato.median():,.0f}")

print("\nPercentiles:")
print(f"P75: ${valor_contrato.quantile(0.75):,.0f}")
print(f"P90: ${valor_contrato.quantile(0.90):,.0f}")
print(f"P95: ${valor_contrato.quantile(0.95):,.0f}")
print(f"P99: ${valor_contrato.quantile(0.99):,.0f}")

Valor total contractual: $951,907,644,300,466,176
Valor promedio: $45,945,923,559,246
Valor mediano: $72,583,240

Percentiles:
P75: $200,397,562
P90: $1,025,014,314
P95: $2,465,543,666
P99: $30,384,923,191


In [6]:
# Concentración de contratos de alto valor

umbrales = [1e9, 1e10, 1e11, 1e12]

for umbral in umbrales:
    cantidad = (valor_contrato > umbral).sum()
    participacion = cantidad / len(df) * 100
    
    print(
        f"Contratos > ${umbral:,.0f}: "
        f"{cantidad:,} ({participacion:.2f}%)"
    )

Contratos > $1,000,000,000: 2,111 (10.19%)
Contratos > $10,000,000,000: 412 (1.99%)
Contratos > $100,000,000,000: 99 (0.48%)
Contratos > $1,000,000,000,000: 14 (0.07%)


## 4. Concentración del valor contractual

Dada la elevada dispersión observada, se analiza la concentración del valor contractual en los contratos de mayor cuantía.

Este análisis permite diferenciar entre el comportamiento de la mayoría de los contratos y el peso financiero de los contratos de mayor valor.

In [7]:
# Concentración del valor contractual en los contratos de mayor valor

df_valor = df.sort_values("valor_del_contrato", ascending=False).copy()

valor_total = df_valor["valor_del_contrato"].sum()

for n in [1, 5, 10, 50, 100]:
    valor_top = df_valor.head(n)["valor_del_contrato"].sum()
    participacion = valor_top / valor_total * 100
    
    print(
        f"Top {n}: "
        f"${valor_top:,.0f} "
        f"({participacion:.2f}% del valor contractual total)"
    )

Top 1: $921,600,000,000,000,000 (96.82% del valor contractual total)
Top 5: $943,486,634,828,999,936 (99.12% del valor contractual total)
Top 10: $951,573,905,728,999,936 (99.96% del valor contractual total)
Top 50: $951,881,930,261,846,656 (100.00% del valor contractual total)
Top 100: $951,891,789,818,103,936 (100.00% del valor contractual total)


## 5. Comportamiento financiero de los contratos

A continuación se analiza el nivel de ejecución y pago de los contratos a partir de los indicadores financieros construidos durante la etapa de alistamiento.

El análisis busca identificar el comportamiento general de estos indicadores y posibles diferencias entre ejecución contractual y pagos realizados.

In [8]:
# Estadísticos de ejecución y pago

df[[
    "porcentaje_ejecutado",
    "porcentaje_pagado"
]].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

,porcentaje_ejecutado,porcentaje_pagado
count,19632.000000,19632.000000
mean,34.731067,34.056854
std,43.151165,43.136797
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,87.329793,87.169206
90%,99.369898,99.360966
95%,100.000000,100.000000
99%,100.000000,100.000000


In [9]:
# Clasificación del nivel de ejecución

def clasificar_ejecucion(valor):
    if pd.isna(valor):
        return "Sin dato"
    elif valor == 0:
        return "0%"
    elif valor < 50:
        return "1% - 49%"
    elif valor < 100:
        return "50% - 99%"
    else:
        return ">= 100%"

df["nivel_ejecucion"] = df["porcentaje_ejecutado"].apply(clasificar_ejecucion)

ejecucion_nivel = (
    df["nivel_ejecucion"]
    .value_counts()
    .reindex(["0%", "1% - 49%", "50% - 99%", ">= 100%", "Sin dato"])
    .fillna(0)
    .astype(int)
    .rename_axis("nivel_ejecucion")
    .reset_index(name="contratos")
)

ejecucion_nivel

,nivel_ejecucion,contratos
0,0%,11310
1,1% - 49%,939
2,50% - 99%,6098
3,>= 100%,1285
4,Sin dato,1086


## 6. Ejecución financiera según estado contractual

Para contextualizar los niveles de ejecución observados, se analiza su comportamiento según el estado del contrato.

Este cruce permite diferenciar los contratos con baja o nula ejecución según su situación contractual y evitar interpretar de manera aislada un porcentaje de ejecución igual a cero.

In [10]:
# Ejecución financiera según estado contractual

ejecucion_estado = (
    df.groupby("estado_contrato")
    .agg(
        contratos=("id_contrato", "count"),
        ejecucion_promedio=("porcentaje_ejecutado", "mean"),
        ejecucion_mediana=("porcentaje_ejecutado", "median")
    )
    .sort_values("contratos", ascending=False)
)

ejecucion_estado

,contratos,ejecucion_promedio,ejecucion_mediana
estado_contrato,,,
Cerrado,7622,65.683971,88.716434
Modificado,4132,8.599017,0.000000
terminado,2946,25.412307,0.000000
Borrador,2131,0.000000,0.000000
En ejecución,1720,40.922830,48.893939
Aprobado,773,3.569662,0.000000
Cancelado,578,0.000000,0.000000
En aprobación,283,0.000000,0.000000
enviado Proveedor,250,0.000000,0.000000


In [11]:
# Proporción de contratos con ejecución igual a 0% por estado

ejecucion_cero_estado = (
    df.groupby("estado_contrato")
    .agg(
        contratos=("id_contrato", "count"),
        contratos_0_ejecucion=(
            "porcentaje_ejecutado",
            lambda x: (x == 0).sum()
        )
    )
)

ejecucion_cero_estado["porcentaje_0_ejecucion"] = (
    ejecucion_cero_estado["contratos_0_ejecucion"]
    / ejecucion_cero_estado["contratos"]
    * 100
)

ejecucion_cero_estado.sort_values(
    "contratos",
    ascending=False
)

,contratos,contratos_0_ejecucion,porcentaje_0_ejecucion
estado_contrato,,,
Cerrado,7622,1815,23.812648
Modificado,4132,3576,86.544046
terminado,2946,2053,69.687712
Borrador,2131,1586,74.425153
En ejecución,1720,558,32.441860
Aprobado,773,694,89.780078
Cancelado,578,308,53.287197
En aprobación,283,275,97.173145
enviado Proveedor,250,220,88.000000


## 7. Relación entre ejecución y pago

Se compara el porcentaje de ejecución financiera con el porcentaje pagado para identificar si ambos indicadores presentan un comportamiento similar.

Esta comparación permite identificar posibles diferencias entre el avance financiero registrado y los pagos efectuados.

In [12]:
# Comparación entre ejecución y pago

df[[
    "porcentaje_ejecutado",
    "porcentaje_pagado"
]].corr()

,porcentaje_ejecutado,porcentaje_pagado
porcentaje_ejecutado,1.000000,0.989179
porcentaje_pagado,0.989179,1.000000


In [13]:
# Diferencia entre ejecución y pago

df["diferencia_ejecucion_pago"] = (
    df["porcentaje_ejecutado"] -
    df["porcentaje_pagado"]
)

df["diferencia_ejecucion_pago"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)

count    19632.000000
mean         0.674213
std          6.346980
min          0.000000
10%          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
90%          0.000000
95%          0.000000
max        100.000000
Name: diferencia_ejecucion_pago, dtype: float64

## 8. Extensiones contractuales

Se analiza la presencia de extensiones contractuales y su distribución según el tipo de contrato.

El objetivo es identificar los segmentos contractuales donde las extensiones presentan mayor frecuencia y establecer si existen concentraciones relevantes para el análisis posterior.

In [14]:
# Extensiones por tipo de contrato

extension_tipo = (
    df.groupby("tipo_de_contrato")
    .agg(
        contratos=("id_contrato", "count"),
        contratos_con_extension=("tiene_extension", lambda x: (x == "Sí").sum())
    )
)

extension_tipo["porcentaje_con_extension"] = (
    extension_tipo["contratos_con_extension"]
    / extension_tipo["contratos"]
    * 100
)

extension_tipo.sort_values(
    "contratos_con_extension",
    ascending=False
)

,contratos,contratos_con_extension,porcentaje_con_extension
tipo_de_contrato,,,
Otro,2326,1548,66.552021
Obra,3409,711,20.856556
Prestación de servicios,12058,617,5.116935
Interventoría,1530,612,40.000000
Consultoría,573,186,32.460733
Suministros,593,31,5.227656
Comodato,74,10,13.513514
Compraventa,81,8,9.876543
Arrendamiento de inmuebles,31,4,12.903226


## 9. Extensiones y pendiente de ejecución

Se analiza la relación entre la existencia de extensiones contractuales y el valor pendiente de ejecución.

El objetivo es identificar si los contratos que presentan extensiones concentran mayores valores pendientes de ejecución y establecer un grupo de contratos que pueda requerir especial atención en el tablero.

In [15]:
# Pendiente de ejecución según existencia de extensión

pendiente_extension = (
    df.groupby("tiene_extension")
    .agg(
        contratos=("id_contrato", "count"),
        pendiente_total=("valor_pendiente_de_ejecucion", "sum"),
        pendiente_promedio=("valor_pendiente_de_ejecucion", "mean"),
        pendiente_mediana=("valor_pendiente_de_ejecucion", "median")
    )
)

pendiente_extension

,contratos,pendiente_total,pendiente_promedio,pendiente_mediana
tiene_extension,,,,
No,16990,2.875720e+13,1.692595e+09,21000000.0
Sí,3728,9.316423e+12,2.499041e+09,201620930.0


In [18]:
df["contrato_prioritario"].value_counts(dropna=False)

contrato_prioritario
No    19968
Sí      750
Name: count, dtype: int64

In [19]:
# Extensiones dentro de los contratos prioritarios

prioritarios_extension = (
    df[df["contrato_prioritario"] == "Sí"]
    ["tiene_extension"]
    .value_counts()
)

prioritarios_extension

tiene_extension
Sí    750
Name: count, dtype: int64

In [20]:
# Porcentaje de contratos prioritarios con extensión

total_prioritarios = (
    df["contrato_prioritario"] == "Sí"
).sum()

prioritarios_con_extension = (
    (df["contrato_prioritario"] == "Sí") &
    (df["tiene_extension"] == "Sí")
).sum()

print(f"Contratos prioritarios: {total_prioritarios:,}")
print(f"Con extensión: {prioritarios_con_extension:,}")
print(
    f"Porcentaje con extensión: "
    f"{prioritarios_con_extension / total_prioritarios * 100:.2f}%"
)

Contratos prioritarios: 750
Con extensión: 750
Porcentaje con extensión: 100.00%


## 10. Distribución de los contratos prioritarios

Se analiza la distribución de los contratos prioritarios según el tipo de contrato y la modalidad de contratación.

Los contratos prioritarios corresponden a aquellos que superan simultáneamente los percentiles 90 de valor pendiente de ejecución y días adicionados.

El objetivo es identificar las categorías contractuales donde se concentra este grupo y orientar las visualizaciones y filtros del tablero hacia las dimensiones con mayor relevancia.

In [21]:
# Contratos prioritarios por tipo de contrato

prioritarios_tipo = (
    df[df["contrato_prioritario"] == "Sí"]
    ["tipo_de_contrato"]
    .value_counts()
    .to_frame("contratos")
)

prioritarios_tipo

,contratos
tipo_de_contrato,
Obra,274
Otro,154
Prestación de servicios,134
Interventoría,108
Consultoría,75
Compraventa,4
Suministros,1


In [22]:
# Participación de cada tipo dentro de los contratos prioritarios

prioritarios_tipo["porcentaje"] = (
    prioritarios_tipo["contratos"]
    / prioritarios_tipo["contratos"].sum()
    * 100
)

prioritarios_tipo

,contratos,porcentaje
tipo_de_contrato,,
Obra,274,36.533333
Otro,154,20.533333
Prestación de servicios,134,17.866667
Interventoría,108,14.400000
Consultoría,75,10.000000
Compraventa,4,0.533333
Suministros,1,0.133333


In [23]:
# Contratos prioritarios por modalidad de contratación

prioritarios_modalidad = (
    df[df["contrato_prioritario"] == "Sí"]
    ["modalidad_de_contratacion"]
    .value_counts()
    .to_frame("contratos")
)

prioritarios_modalidad["porcentaje"] = (
    prioritarios_modalidad["contratos"]
    / prioritarios_modalidad["contratos"].sum()
    * 100
)

prioritarios_modalidad

,contratos,porcentaje
modalidad_de_contratacion,,
Contratación directa,322,42.933333
Concurso de méritos abierto,183,24.400000
Licitación pública,119,15.866667
Licitación pública Obra Publica,89,11.866667
Selección Abreviada de Menor Cuantía,18,2.400000
Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes,10,1.333333
Selección abreviada subasta inversa,7,0.933333
Contratación Directa (con ofertas),2,0.266667


## Cierre de la exploración y análisis

La exploración permitió caracterizar el comportamiento de la ejecución financiera y las extensiones contractuales de los contratos analizados.

Se identificó una alta concentración de contratos con niveles bajos de ejecución financiera, así como una fuerte asociación entre los porcentajes ejecutado y pagado. Adicionalmente, los contratos que presentan extensiones muestran una pendiente de ejecución mediana superior a aquellos sin extensión.

Como mecanismo de priorización para el seguimiento, se identificaron 750 contratos que superan simultáneamente el P90 del valor pendiente de ejecución y el P90 de los días adicionados. El 100 % de estos contratos presenta extensión contractual.

Los contratos prioritarios presentan además una concentración importante por tipo de contrato y modalidad de contratación, lo que permitirá orientar los filtros y visualizaciones del tablero.

Estos hallazgos constituyen la base para el diseño del tablero de la Pregunta 3, cuyo propósito será facilitar la identificación y seguimiento de los contratos con mayores niveles de recursos pendientes y extensiones contractuales.